In [ ]:
import jax
import jax.numpy as jnp
import functools
import pickle
from dataclasses import dataclass
import dataclasses

@dataclass
class BlockParams:
    W_q: jax.Array
    W_k: jax.Array
    W_v: jax.Array
    W_1: jax.Array
    W_2: jax.Array
    gamma1: jax.Array
    beta1: jax.Array
    gamma2: jax.Array
    beta2: jax.Array
    n_heads: int = 1
    embed_dim: int = 256
    hidden_dim: int = 1024

    @classmethod
    def random_init(cls, key: jax.Array, embed_dim: int, hidden_dim: int, n_heads: int) -> 'BlockParams':
        def init(key: jax.Array, shape: tuple[int, ...]) -> jax.Array:
            scale = 1.0/shape[0]
            return jax.random.normal(key, shape) * scale
        
        shapes = [
        (embed_dim, embed_dim),    # W_q
        (embed_dim, embed_dim),    # W_k
        (embed_dim, embed_dim),    # W_v
        (embed_dim, hidden_dim),   # W_1
        (hidden_dim, embed_dim),   # W_2
        ]
        keys = jax.random.split(key, len(shapes))
        W_q, W_k, W_v, W_1, W_2 = [init(k, s) for k, s in zip(keys, shapes)]
        return cls(
        W_q, W_k, W_v, W_1, W_2,
        gamma1=jnp.ones(embed_dim),  beta1=jnp.zeros(embed_dim),
        gamma2=jnp.ones(embed_dim),  beta2=jnp.zeros(embed_dim),
        n_heads=n_heads, embed_dim=embed_dim, hidden_dim=hidden_dim,
        )
        


jax.tree_util.register_dataclass(
    BlockParams,
    data_fields=['W_q', 'W_k', 'W_v', 'W_1', 'W_2',
                 'gamma1', 'beta1', 'gamma2', 'beta2'],
    meta_fields=['n_heads','embed_dim','hidden_dim'],
)

@jax.jit
def compute_qkv(X: jax.Array, W_q: jax.Array, W_k: jax.Array, W_v: jax.Array) -> tuple[jax.Array, jax.Array, jax.Array]:
    Q = jnp.dot(X, W_q)
    K = jnp.dot(X, W_k)
    V = jnp.dot(X, W_v)
    return Q, K, V

@jax.jit
def compute_attention(Q: jax.Array, K: jax.Array, V: jax.Array) -> jax.Array:
    d_k = Q.shape[-1]
    QK = jnp.matmul(Q, jnp.swapaxes(K, -1, -2)) / jnp.sqrt(d_k)
    attention_weights = jax.nn.softmax(QK)
    attention_output = jnp.matmul(attention_weights, V)
    return attention_output


def rope_tables(seq_len: int, embed_dim: int, base: float = 10000.0) -> tuple[jax.Array, jax.Array]:
    inv_freq = base ** (-jnp.arange(0, embed_dim, 2) / embed_dim)  # (d/2,)
    angles   = jnp.outer(jnp.arange(seq_len), inv_freq)           # (seq, d/2)
    return jnp.cos(angles), jnp.sin(angles)

@jax.jit
def apply_rope(x: jax.Array, cos: jax.Array, sin: jax.Array) -> jax.Array:
    # x: (..., seq, embed_dim), pairs adjacent dims (0,1),(2,3),...
    x1, x2 = x[..., 0::2], x[..., 1::2]
    out = jnp.stack([x1 * cos - x2 * sin,
                     x1 * sin + x2 * cos], axis=-1)
    return out.reshape(x.shape)


@jax.jit
def compute_masked_attention(Q: jax.Array, K: jax.Array, V: jax.Array) -> jax.Array:
    d_k = Q.shape[-1]
    QK = jnp.matmul(Q, jnp.swapaxes(K, -1, -2)) / jnp.sqrt(d_k)
    causal_mask = -jnp.inf*jnp.ones(QK.shape)
    causal_mask = jnp.triu(causal_mask, k=1) 
    attention_weights = jax.nn.softmax(QK + causal_mask)
    attention_output = jnp.matmul(attention_weights, V)
    return attention_output

@jax.jit
def layer_norm(X: jax.Array, gamma: jax.Array, beta: jax.Array) -> jax.Array:
    mean = jnp.mean(X, axis=-1, keepdims=True)
    var = jnp.var(X, axis=-1, keepdims=True)
    return gamma * (X - mean) / jnp.sqrt(var + 1e-5) + beta

@jax.jit
def relu(X: jax.Array) -> jax.Array:
    return jnp.maximum(0, X)

@jax.jit
def feed_forward(X: jax.Array, W_1: jax.Array, W_2: jax.Array) -> jax.Array:
    return jnp.dot(relu(jnp.dot(X, W_1)), W_2)

@jax.jit
def transformer_block(X: jax.Array, params: BlockParams) -> jax.Array:
    Q, K, V = compute_qkv(layer_norm(X, params.gamma1, params.beta1), params.W_q, params.W_k, params.W_v)
    seq = X.shape[-2]
    Q = apply_rope(Q, COS[:seq], SIN[:seq])
    K = apply_rope(K, COS[:seq], SIN[:seq])
    X = X + compute_masked_attention(Q, K, V)
    X = X + feed_forward(layer_norm(X, params.gamma2, params.beta2), params.W_1, params.W_2)
    return X


@dataclass
class ModelParams:
    W_embed: jax.Array
    blocks: list[BlockParams]
    W_decode: jax.Array
    vocab_size: int = 13
    n_blocks: int = 1
    embed_dim: int = 256
    hidden_dim: int = 1024

    @classmethod
    def random_init(cls, key: jax.Array, vocab_size: int, n_blocks: int, embed_dim: int, hidden_dim: int, n_heads: int) -> 'ModelParams':
        k_embed, k_decode, k_blocks = jax.random.split(key, 3)
        W_embed  = jax.random.normal(k_embed,  (vocab_size, embed_dim)) / jnp.sqrt(embed_dim)
        W_decode = jax.random.normal(k_decode, (embed_dim, vocab_size)) / jnp.sqrt(embed_dim)
        blocks   = [BlockParams.random_init(k, embed_dim, hidden_dim, n_heads)
                    for k in jax.random.split(k_blocks, n_blocks)]
        return cls(W_embed, blocks, W_decode, vocab_size, n_blocks, embed_dim, hidden_dim)


jax.tree_util.register_dataclass(
    ModelParams,
    data_fields=['W_embed', 'blocks', 'W_decode'],
    meta_fields=['vocab_size', 'n_blocks', 'embed_dim', 'hidden_dim'],
)

@jax.jit
def model_forward(model_params: ModelParams, X: jax.Array) -> jax.Array:
    X = model_params.W_embed[X]
    for block in model_params.blocks:
        X = transformer_block(X, block)
    return X @ model_params.W_decode

@jax.jit
def cross_entropy_loss(logits: jax.Array, targets: jax.Array) -> jax.Array:
    logp = jax.nn.log_softmax(logits, -1)
    return -jnp.mean(jnp.take_along_axis(logp, targets[..., None], -1))

@jax.jit
def masked_cross_entropy_loss(logits: jax.Array, targets: jax.Array, mask: jax.Array) -> tuple[jax.Array, jax.Array]:
    logp = jax.nn.log_softmax(logits, -1)
    nll = -jnp.take_along_axis(logp, targets[..., None], -1).squeeze(-1) #selects target index across log probs
    nll = nll*mask
    count = mask.sum()*targets.shape[-1]
    return nll.sum()/(count), count

@jax.jit
def loss(params: ModelParams, X: jax.Array, targets: jax.Array) -> jax.Array:
    logits = model_forward(params, X)
    return cross_entropy_loss(logits, targets)

@jax.jit
def masked_loss(params: ModelParams, X: jax.Array, targets: jax.Array, mask: jax.Array) -> tuple[jax.Array, jax.Array]:
    logits = model_forward(params, X)
    loss, count = masked_cross_entropy_loss(logits, targets, mask)
    return (loss, count)

@jax.jit
def train_step(params: ModelParams, X: jax.Array, Y: jax.Array, mask: jax.Array, optim_state: AdamWState, lr: jax.Array) -> tuple[tuple[ModelParams, AdamWState], tuple[jax.Array, jax.Array]]:
    (loss_val, count), grads = jax.value_and_grad(masked_loss, has_aux=True)(params, X, Y, mask)
    params, optim_state = AdamW_optim_step(params, grads, optim_state, lr)
    return (params, optim_state), (loss_val, count)

@jax.jit
def train_batch_step(carry: tuple[ModelParams, AdamWState, jax.Array], xs: tuple[jax.Array, jax.Array]) -> tuple[tuple[ModelParams, AdamWState, jax.Array], tuple[jax.Array, jax.Array]]:
    params, optim_state, lr = carry
    batches, mask = xs
    X, Y = batches[:, :-1], batches[:, 1:]   
    (params, optim_state), out = train_step(params, X, Y, mask, optim_state, lr)
    return (params, optim_state, lr), out

@jax.jit
def train_epoch_step(params: ModelParams, optim_state: AdamWState, batches: jax.Array, masks: jax.Array, lr: jax.Array) -> tuple[ModelParams, AdamWState, jax.Array]:
    (params, optim_state, lr), (batch_losses, batch_counts) = jax.lax.scan(
        train_batch_step, (params, optim_state, lr), (batches, masks))
    epoch_loss = jnp.sum(batch_losses * batch_counts) / jnp.sum(batch_counts)   # exact 
    return params, optim_state, epoch_loss

@jax.jit
def valid_step(params: ModelParams, X: jax.Array, Y: jax.Array, mask: jax.Array) -> tuple[jax.Array, jax.Array]:
    loss_val, count = masked_loss(params, X, Y, mask)
    return loss_val, count

@jax.jit
def valid_batch_step(params: ModelParams, xs: tuple[jax.Array, jax.Array]) -> tuple[ModelParams, tuple[jax.Array, jax.Array]]:
    batches, mask = xs
    X, Y = batches[:, :-1], batches[:, 1:]   
    loss_val, count = valid_step(params, X, Y, mask)
    return params, (loss_val, count) 

@jax.jit
def valid_epoch_step(params: ModelParams, batches: jax.Array, masks: jax.Array) -> jax.Array:
    step = lambda p, xs: valid_batch_step(p, xs)
    params, (batch_losses, batch_counts) = jax.lax.scan(step, params, (batches, masks))
    epoch_loss = jnp.sum(batch_losses * batch_counts) / jnp.sum(batch_counts)  
    return epoch_loss


@dataclass
class AdamWState:
    m: ModelParams
    v: ModelParams
    t: jax.Array
    b1: float = 0.9
    b2: float = 0.999
    eps: float = 1e-9
    wd: float = 0.01

    @classmethod
    def init(cls, params: ModelParams, **hparams: float) -> 'AdamWState':
        return cls(m=jax.tree.map(jnp.zeros_like, params),
              v=jax.tree.map(jnp.zeros_like, params),
              t=jnp.array(0, jnp.int32),
              **hparams,)

jax.tree_util.register_dataclass(
    AdamWState,
    data_fields=['m','v','t'],
    meta_fields=['b1','b2','eps','wd'],
)

@jax.jit
def AdamW_optim_step(params: ModelParams, grad: ModelParams, optim_state: AdamWState, lr: jax.Array) -> tuple[ModelParams, AdamWState]:
    _m,_v,_t = optim_state.m, optim_state.v, optim_state.t
    b1, b2, eps, wd = optim_state.b1, optim_state.b2, optim_state.eps, optim_state.wd

    t = _t + 1
    m = jax.tree.map(lambda _m, g: b1*_m + (1-b1)*g, _m, grad)
    v = jax.tree.map(lambda _v, g: b2*_v + (1-b2)*g**2, _v, grad)

    bc1 = 1 - b1**t
    bc2 = 1 - b2**t
    mhat = jax.tree.map(lambda m: m / bc1, m)
    vhat = jax.tree.map(lambda v: v / bc2, v)

    def weight_update(params: jax.Array, mhat: jax.Array, vhat: jax.Array, eps: float, wd: float, lr: jax.Array) -> jax.Array:
        return params - lr * (mhat / (jnp.sqrt(vhat)+eps) + wd*params)

    f = lambda p, m, v: weight_update(p, m, v, eps, wd, lr)
    params =  jax.tree.map(f, params, mhat, vhat) 
    return params, dataclasses.replace(optim_state, m=m, v=v, t=t)

@jax.jit
def cosine_lr_schudule(t: jax.Array | int, peak: float, warmup: int, end: float, total: int) -> jax.Array:
    warm  = peak * t / warmup
    prog  = jnp.clip((t - warmup) / (total - warmup), 0.0, 1.0)
    cos   = 0.5 * (1.0 + jnp.cos(jnp.pi * prog))
    decay = end + (peak - end) * cos
    return jnp.where(t < warmup, warm, decay)  


EMBED_DIM    = 256
ROPE_MAX_SEQ = 64

COS, SIN = rope_tables(ROPE_MAX_SEQ, EMBED_DIM)  

key = jax.random.key(42)
tokenizer = {c: i for i, c in enumerate("0123456789+= ")}
params = ModelParams.random_init(key, vocab_size=len(tokenizer), n_blocks=16,
                                 embed_dim=256, hidden_dim=1024, n_heads=1)


string = "2+2="
print(string)
tokens = jnp.array([tokenizer[c] for c in string])
print(tokens)
logits = model_forward(params, tokens)

probs = jax.nn.softmax(logits, axis=-1)        
preds = jnp.argmax(logits, axis=-1)
print(preds)

optim_state = AdamWState.init(params)

print(optim_state)

In [2]:
data = []
import random
random.seed(42)
digits = 3
mod = 113
tokens_len = digits*3+3
for a in range(0,10**3):
    for b in range(0,10**3):
        s = (str(a)+"+"+str(b)+"="+str((a+b)%mod)).ljust(tokens_len)   # pad to length 9 with spaces
        tokens = [tokenizer[c] for c in s]
        data.append(tokens)

print(len(data))
training_portion = 0.05 
valid_portion = 0.025
random.shuffle(data)
slice = int(len(data)*training_portion)
valid_slice = int(len(data)*valid_portion)

train = jnp.array(data[0:slice])
test = jnp.array(data[slice:-valid_slice])
valid = jnp.array(data[-valid_slice:])
print(len(train),len(test), len(valid))


1000000
50000 925000 25000


In [3]:
import jax, shutil
import os
from google.colab import drive
drive.mount('/content/drive')

DB_LOCAL = "/content/mlflow.db"                    # fast, reliable VM-local disk
DB_DRIVE = "/content/drive/MyDrive/mlflow.db"      # synced copy your laptop reads
if os.path.exists(DB_DRIVE):                       # resume existing history if present
    shutil.copy(DB_DRIVE, DB_LOCAL)

params = ModelParams.random_init(key, vocab_size=len(tokenizer), n_blocks=2,
                                 embed_dim=256, hidden_dim=1024//2, n_heads=1)

optim_state = AdamWState.init(params)

peak = 1e-5
warmup = 1000
end = 3e-7
total = 20

batch_size = 1024*8

if len(train) % batch_size != 0:
    tail_size = -len(train) % batch_size
    train_padded = jnp.pad(train, ((0, tail_size),(0,0)))
    train_mask = jnp.ones(len(train_padded)).at[len(train_padded)-tail_size:].set(0)
else:
    train_mask = jnp.ones(len(train))
    train_padded = train

n_train_batches = len(train_padded)//batch_size 

train_batches = train_padded[:n_train_batches*batch_size].reshape(n_train_batches, batch_size, -1) 
train_masks = train_mask[:n_train_batches*batch_size].reshape(n_train_batches, batch_size, -1) 

#first jit compilation
i=1
lr = cosine_lr_schudule(i, peak, warmup, end, total)
params, optim_state, epoch_loss = train_epoch_step(params, optim_state, train_batches, train_masks, lr)
epoch_loss.block_until_ready()

jax.profiler.start_trace("/content/jax-trace")
for i in range(2):
    lr = cosine_lr_schudule(i, peak, warmup, end, total)
    params, optim_state, epoch_loss = train_epoch_step(params, optim_state, train_batches, train_masks, lr)
epoch_loss.block_until_ready()
jax.profiler.stop_trace()

shutil.make_archive("/content/jax-trace", "gztar", "/content/jax-trace")
shutil.copy("/content/jax-trace.tar.gz", "/content/drive/MyDrive/jax-trace.tar.gz")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


'/content/drive/MyDrive/jax-trace.tar.gz'

In [32]:
EQ = tokenizer['=']
SPACE = tokenizer[' ']
@jax.jit
def answer_accuracy(logits, X, targets, mask):
    after_eq   = jnp.cumsum(X == EQ, axis=-1) > 0          # positions after the '='
    before_pad = jnp.cumsum(targets == SPACE, axis=-1) == 0  # before first padding space
    is_answer  = after_eq & before_pad                     # answer digits only: (after '=') .. (first ' ' / end)
    pos_mask   = is_answer * mask
    preds      = jnp.argmax(logits, axis=-1)
    correct    = (preds == targets) * pos_mask
    return correct.sum(), pos_mask.sum()  

@jax.jit
def data_step(params, X, Y, mask):
    logits = model_forward(params, X)
    acc_count, total_count = answer_accuracy(logits, X, Y, mask)
    return acc_count, total_count

@jax.jit
def data_batch_step(params, xs):
    batches, mask = xs
    X, Y = batches[:, :-1], batches[:, 1:]   
    acc_count, total_count = data_step(params, X, Y, mask)
    return params, (acc_count, total_count) 

@jax.jit
def data_epoch_step(params, batches, masks):
    step = lambda p, xs: data_batch_step(p, xs)
    params, (batch_accs, batch_counts) = jax.lax.scan(step, params, (batches, masks))
    accuracy = jnp.sum(batch_accs)/jnp.sum(batch_counts)  
    return accuracy

In [5]:
!pip install -U "numpy>=2.1" scipy
!pip install -q mlflow --ignore-installed blinker


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.55.1 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
numba 0.65.1 requires numpy<2.5,>=1.22, but you have numpy 2.5.0 which is incompatible.


In [33]:
lr = 3e-4
epochs = 1000
batch_size = 1024*8
from tqdm import tqdm
import mlflow, datetime, shutil, os, time

# --- SQLite backend, synced via Google Drive ---
# SQLite can't be written directly on the Drive FUSE mount (locking unsupported),
# so we work on a local-disk db and copy it to/from Drive as a single file.
from google.colab import drive
drive.mount('/content/drive')

DB_LOCAL = "/content/mlflow.db"                    # fast, reliable VM-local disk
DB_DRIVE = "/content/drive/MyDrive/mlflow.db"      # synced copy your laptop reads
if os.path.exists(DB_DRIVE):                       # resume existing history if present
    shutil.copy(DB_DRIVE, DB_LOCAL)

mlflow.set_tracking_uri(f"sqlite:///{DB_LOCAL}")
mlflow.set_experiment(f"addition-transformer-{digits}-digits-{mod}-mod")      # named container; runs show up here
mlflow.config.enable_async_logging()

EMBED_DIM    = 256
ROPE_MAX_SEQ = 64

COS, SIN = rope_tables(ROPE_MAX_SEQ, EMBED_DIM)  


key = jax.random.key(42)
tokenizer = {c: i for i, c in enumerate("0123456789+= ")}
params = ModelParams.random_init(key, vocab_size=len(tokenizer), n_blocks=2,
                                 embed_dim=256, hidden_dim=1024//2, n_heads=1)

optim_state = AdamWState.init(params)

peak = 3e-4
warmup = 50
end = 3e-7
total = epochs

if len(train) % batch_size != 0:
    tail_size = -len(train) % batch_size
    train_padded = jnp.pad(train, ((0, tail_size),(0,0)))
    train_mask = jnp.ones(len(train_padded)).at[len(train_padded)-tail_size:].set(0)
else:
    train_mask = jnp.ones(len(train))
    train_padded = train

if len(valid) % batch_size != 0:
    tail_size = -len(valid) % batch_size
    valid_padded = jnp.pad(valid, ((0, tail_size),(0,0)))
    valid_mask = jnp.ones(len(valid_padded)).at[len(valid_padded)-tail_size:].set(0)
else:
    valid_mask = jnp.ones(len(valid))
    valid_padded = valid

n_train_batches = len(train_padded)//batch_size 

train_batches = train_padded[:n_train_batches*batch_size].reshape(n_train_batches, batch_size, -1) 
train_masks = train_mask[:n_train_batches*batch_size].reshape(n_train_batches, batch_size, -1) 

n_valid_batches = len(valid_padded)//batch_size 

valid_batches = valid_padded[:n_valid_batches*batch_size].reshape(n_valid_batches, batch_size, -1) 
valid_masks = valid_mask[:n_valid_batches*batch_size].reshape(n_valid_batches, batch_size, -1) 

print("train batches", train_batches.shape,"mask", train_masks.shape)
print("valid batches", valid_batches.shape,"mask", valid_masks.shape)

run_name = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
dev = jax.devices()[0]
buf = []

with mlflow.start_run(run_name=run_name):
    mlflow.log_params({"lr": lr, "epochs": epochs, "batch_size": batch_size,
                       "n_blocks": params.n_blocks, "embed_dim": params.embed_dim})

    n_examples = train_batches.shape[0] * train_batches.shape[1]
    for i in tqdm(range(epochs), total=epochs):
        t0 = time.perf_counter()
        lr = cosine_lr_schudule(i, peak, warmup, end, total)
        params, optim_state, epoch_loss = train_epoch_step(params, optim_state, train_batches, train_masks, lr)
        valid_loss = valid_epoch_step(params, valid_batches, valid_masks)
        train_acc = data_epoch_step(params, train_batches, train_masks)
        valid_acc = data_epoch_step(params, valid_batches, valid_masks)


        buf.append((i, epoch_loss, valid_loss, train_acc, valid_acc))
        if i %25 == 0:
            steps, train_losses, valid_losses, train_accs, valid_accs = zip(*buf); buf = []
            train_losses = jax.device_get(jnp.stack(train_losses))
            valid_losses = jax.device_get(jnp.stack(valid_losses))
            train_accs   = jax.device_get(jnp.stack(train_accs))
            valid_accs   = jax.device_get(jnp.stack(valid_accs))

            for s, tl, vl, ta, va in zip(steps, train_losses, valid_losses, train_accs, valid_accs):
                mlflow.log_metrics({
                    "train_loss":     float(tl),
                    "valid_loss":     float(vl),
                    "train_accuracy": float(ta),
                    "valid_accuracy": float(va),
                }, step=s)
            mem = dev.memory_stats()
            dt  = time.perf_counter() - t0
            mlflow.log_metrics({                 
                "lr":               float(lr),
                "examples_per_sec": n_examples / dt,
                "tpu_mem_used_gb":  mem["bytes_in_use"]      / 1e9,
                "tpu_mem_peak_gb":  mem["peak_bytes_in_use"] / 1e9,
            }, step=i)
    ckpt_path = "/content/model_params.pkl"
    with open(ckpt_path, "wb") as f:
        pickle.dump(jax.device_get(params), f) 
    mlflow.log_artifact(ckpt_path, artifact_path="checkpoints")
    shutil.copy(ckpt_path, "/content/drive/MyDrive/model_params.pkl")

shutil.copy(DB_LOCAL, DB_DRIVE)                     # push updated db back to Drive
print("synced ->", DB_DRIVE)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train batches (13, 8192, 12) mask (13, 8192, 1)
valid batches (4, 8192, 12) mask (4, 8192, 1)


100%|██████████| 1000/1000 [07:50<00:00,  2.13it/s]


synced -> /content/drive/MyDrive/mlflow.db


In [ ]:
print(jax.devices())

[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]


In [ ]:
inv = {i: c for c, i in tokenizer.items()}
a, b = random.randint(0, 10**digits), random.randint(0, 10**digits)
ids = [tokenizer[c] for c in f"{a}+{b}="]
while len(ids) < tokens_len:                                       # generate one token at a time
    nxt = int(jnp.argmax(model_forward(params, jnp.array(ids))[-1]))   # last position's prediction
    ids.append(nxt)
print("target:", f"{a}+{b}={a+b}", "  prediction:", "".join(inv[i] for i in ids))


target: 651+892=1543   prediction: 651+892=1543


In [ ]:
data = jnp.array(data)
batch_size = 1024

if len(data) % batch_size != 0:
    tail_size = -len(valid) % batch_size
    data_padded = jnp.pad(data, ((0, tail_size),(0,0)))
    data_mask = jnp.ones(len(data_padded)).at[len(data_padded)-tail_size:].set(0)
else:
    data_mask = jnp.ones(len(data))
    data_padded = data

n_data_batches = len(data_padded)//batch_size 

data_batches = data_padded[:n_data_batches*batch_size].reshape(n_data_batches, batch_size, -1) 
data_masks = data_mask[:n_data_batches*batch_size].reshape(n_data_batches, batch_size, -1) 

accuracy = data_epoch_step(params, data_batches, data_masks)
print(accuracy)


0.8080176
